## Using propaq

Welcome to the propaq user guide! This notebook will help you get started with using propaq for quantum circuit simulation. This covers the basics of propaq, we encourage the user to explore the documentation and other notebooks for more advanced features.

## Basic usage

In order to run a Heisenberg simulation with propaq, you will need a circuit, observable, and a state. For this example, we'll be using Qiskit to create a simple circuit and observable.

In [2]:
from qiskit import QuantumCircuit
from qiskit.quantum_info import SparsePauliOp

from qiskit.circuit.library import (
    XXPlusYYGate,
    PhaseGate,
    RZGate,
    CPhaseGate,
    SwapGate,
    XGate
) 
import numpy as np

GATES = [
    (lambda: XXPlusYYGate(
        np.random.uniform(0, 2 * np.pi),
        np.random.uniform(0, 2 * np.pi)
    ), 2),
    (lambda: PhaseGate(np.random.uniform(0, 2 * np.pi)), 1),
    (lambda: RZGate(np.random.uniform(0, 2 * np.pi)), 1),
    (lambda: CPhaseGate(np.random.uniform(0, 2 * np.pi)), 2),
    (lambda: SwapGate(), 2),
    (lambda: XGate(), 1)
]

qc = QuantumCircuit(4)

for _ in range(10): 
    factory, nq = GATES[np.random.randint(len(GATES))]
    gate = factory() 
    qubits = np.random.choice(4, size=nq, replace=False).tolist() 
    qc.append(gate, qubits) 
    
observable = SparsePauliOp.from_list([
    ("XIII", 1.0),
    ("IXII", 1.0),  
    ("IIXI", 1.0),
    ("IIIX", 1.0)
])

Then, we need to convert them into objects recognized by propaq's internals. For this example, we'll implement Majorana propagation.

In [3]:
from propaq.circuits import MajoranaCircuit 
from propaq.datatypes import MajoranaTermSum

mc = MajoranaCircuit.from_qiskit(qc, n_modes = 2 * qc.num_qubits)
mts = MajoranaTermSum.from_sparse_pauli_op(observable)

Now, let's add noise and a truncation strategy, and build the propagator. 

In [4]:
from propaq.noise import UniformNoiseModel, TruncationPolicy

noise = UniformNoiseModel(damping=0.001) # Uniform depolarizing noise with damping parameter 0.001
truncation_policy = TruncationPolicy(
    weight_cutoff=10, # Max weight of terms to keep
    coeff_cutoff=1e-5, # Min coefficient magnitude to keep
    truncation_range=(100_000, 1_000_000) # Only truncate if the number of terms is in this range, or exceeds the upper bound
)

from propaq.propagators import MajoranaPropagator 

prop = MajoranaPropagator(
    noise = noise,
    truncation = truncation_policy,
    n_threads = 4, # Number of threads to use for parallelization
    progress_bar=True
)

Now, we can compute the expectation value of the observable by back-propagating it through the circuit and evaluating the resulting term sum.

In [ ]:
result = prop.expectation_value(mts, mc, initial_state=0)
print("Expectation value:", result.expectation_value)

## Logging 

To gain more information about the propagation process, you can enable logging.

In [6]:
from propaq import Logger, LogParser

In [7]:
logger = Logger(filename="propaq.log", log_every=5) # log every 5 gates

In [8]:
prop_log = MajoranaPropagator(
    noise = noise,
    truncation = truncation_policy,
    progress_bar=True,
    logger = logger
)

In [9]:
result = prop_log.expectation_value(mts, mc, initial_state=0) 

Propagating through gates: 100%|██████████| 21/21 [00:00<00:00, 1829.22it/s, terms=54]


We should now have a file called `propaq.log` in the current directory, which contains JSON lines of the main propagation events. Each line corresponds to either a gate application or a truncation event, and contains relevant information about the event. This is rather unpleasant to read as-is, so we can use the `LogParser` to extract and visualize the information. 

In [10]:
parser = LogParser("propaq.log")

First, let's look at the gate events. This allows us to see how many terms are being generated in the hashmap and outbox at each step of the propagation.

In [11]:
parser.gate_events

[GateEvent(gate_idx=0, layer_idx=0, map_terms=4, outbox_terms=0, avg_ms_per_gate=None, qiskit_gate_idx=9, monomials=None),
 GateEvent(gate_idx=5, layer_idx=2, map_terms=4, outbox_terms=8, avg_ms_per_gate=0.6943862, qiskit_gate_idx=7, monomials=None),
 GateEvent(gate_idx=10, layer_idx=3, map_terms=4, outbox_terms=11, avg_ms_per_gate=0.5610962, qiskit_gate_idx=5, monomials=None),
 GateEvent(gate_idx=15, layer_idx=5, map_terms=4, outbox_terms=14, avg_ms_per_gate=0.4637394, qiskit_gate_idx=1, monomials=None),
 GateEvent(gate_idx=20, layer_idx=5, map_terms=4, outbox_terms=50, avg_ms_per_gate=0.4247318, qiskit_gate_idx=0, monomials=None)]

We can also look at the truncation events, which contain information on the number of terms discarded, the coefficients of the discarded terms, and the truncation thresholds. This can be useful for debugging and tuning the truncation strategy.

In [12]:
parser.truncation_events

[]

The complete list of available logged information is as follows: 

In [13]:
properties = [
    name
    for name, value in LogParser.__dict__.items()
    if isinstance(value, property)
]
properties

['gate_events',
 'truncation_events',
 'surrogate_flush_events',
 'surrogate_flush_deferred_events',
 'surrogate_merge_events',
 'gate_indices',
 'map_terms',
 'outbox_terms',
 'monomials',
 'terms_before',
 'terms_after',
 'terms_discarded',
 'discarded_coeff_l1',
 'discarded_coeff_max',
 'qiskit_gate_indices',
 'avg_ms_per_gate',
 'elapsed_ms',
 'monomials_before',
 'monomials_after',
 'monomials_discarded']

## Cirq

propaq also optionally supports Cirq circuits! This requires installing an extra dependency - 

In [ ]:
!pip install propaq[cirq]

Now, construct an example circuit and observable using Cirq, and follow the same steps as above to compute the expectation value of the observable!

In [15]:
import cirq

CIRQ_GATES = [
    (lambda: cirq.PhasedISwapPowGate(
        phase_exponent=np.random.uniform(0, 2 * np.pi),
        exponent=np.random.uniform(0, 2 * np.pi)
    ), 2),
    (lambda: cirq.ZPowGate(exponent=np.random.uniform(0, 2 * np.pi)), 1),
    (lambda: cirq.CZPowGate(exponent=np.random.uniform(0, 2 * np.pi)), 2),
    (lambda: cirq.SWAP, 2),
    (lambda: cirq.X, 1)
]

qubits = cirq.LineQubit.range(4)
cirq_qc = cirq.Circuit()

for _ in range(10):
    factory, nq = CIRQ_GATES[np.random.randint(len(CIRQ_GATES))]
    gate = factory()
    selected = np.random.choice(4, size=nq, replace=False).tolist()
    cirq_qc.append(gate.on(*[qubits[q] for q in selected]))

cirq_observable = SparsePauliOp.from_list([
    ("XIII", 1.0),
    ("IXII", 1.0),
    ("IIXI", 1.0),
    ("IIIX", 1.0)
])

cirq_mc = MajoranaCircuit.from_cirq(cirq_qc, n_modes=2 * len(qubits))
cirq_mts = MajoranaTermSum.from_sparse_pauli_op(cirq_observable)

cirq_result = prop.expectation_value(cirq_mts, cirq_mc, initial_state=0)
print("Expectation value:", cirq_result.expectation_value)

Propagating through gates: 100%|██████████| 28/28 [00:00<00:00, 1968.53it/s, terms=1890]

Expectation value: -9.921987957160651e-17
